# Example for bssunfold package with Tomography Algorithms

This notebook demonstrates tomographic reconstruction methods adapted for neutron spectrum unfolding:
- `unfold_bsrem`: Block-Sequential Regularized Expectation Maximization
- `unfold_mapem`: Maximum A Posteriori Expectation Maximization
- `unfold_osem`: Ordered Subsets Expectation Maximization
- `unfold_sart`: Simultaneous Algebraic Reconstruction Technique

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bssunfold import Detector, RF_LANL
from bssunfold.utils.plotting import plot_comparison

## Setup detector and reference spectrum

In [ ]:
det = Detector(RF_LANL)
reference_spectrum = pd.read_csv('../tests/MonteCarlo_Calculated_spectra_from_IAEA_Comp_for_comparison.csv')
readings = det.get_effective_readings_for_spectra(reference_spectrum[['E_MeV','ISO_ref_Cf252']])

## Apply tomography-based unfolding methods

In [ ]:
# BSREM method
result_bsrem = det.unfold_bsrem(readings)

# MAP-EM method
result_mapem = det.unfold_mapem(readings)

# OSEM method
result_osem = det.unfold_osem(readings)

# SART method
result_sart = det.unfold_sart(readings)

## Plot comparison of results

In [ ]:
results = {
    "bsrem": result_bsrem,
    "mapem": result_mapem,
    "osem": result_osem,
    "sart": result_sart,
}

fig, ax = plot_comparison(
    results=results,
    readings=readings,
    reference_spectrum=reference_spectrum[["E_MeV", "ISO_ref_Cf252"]],
)

## Calculate and display metrics

In [ ]:
def calculate_metrics(unfolded, reference):
    """Calculate comparison metrics between unfolded and reference spectra."""
    unfolded_vals = unfolded.values
    ref_vals = reference.values
    
    # Normalized root mean square error
    nrmse = np.sqrt(np.mean((unfolded_vals - ref_vals)**2)) / np.mean(ref_vals) * 100
    
    # Mean absolute percentage error
    mape = np.mean(np.abs((ref_vals - unfolded_vals) / (ref_vals + 1e-10))) * 100
    
    # Integral quantity (total fluence)
    integral_unfolded = np.trapz(unfolded_vals, dx=1)
    integral_ref = np.trapz(ref_vals, dx=1)
    integral_ratio = integral_unfolded / integral_ref
    
    return {
        'NRMSE (%)': nrmse,
        'MAPE (%)': mape,
        'Integral Ratio': integral_ratio
    }

reference_values = reference_spectrum['ISO_ref_Cf252'].values
metrics_data = {}

for name, result in results.items():
    metrics_data[name] = calculate_metrics(result['spectrum'], reference_spectrum['ISO_ref_Cf252'])

metrics_df = pd.DataFrame(metrics_data).T
print("\nComparison Metrics:")
print(metrics_df.to_string())

## EURADOS integral-quantity comparison metrics

In [ ]:
from bssunfold.utils.metrics import calculate_dose_rates

# Calculate dose rates for EURADOS comparison
print("\nEURADOS Integral-Quantity Comparison:")
print("="*60)

ref_dose = calculate_dose_rates(reference_spectrum[['E_MeV','ISO_ref_Cf252']], det.E_MeV, det.RF)
print(f"Reference dose rate: {ref_dose:.4f} µSv/h")

for name, result in results.items():
    spectrum_df = pd.DataFrame({'E_MeV': det.E_MeV, 'flux': result['spectrum']})
    calc_dose = calculate_dose_rates(spectrum_df, det.E_MeV, det.RF)
    deviation = (calc_dose - ref_dose) / ref_dose * 100
    print(f"{name:20s}: {calc_dose:.4f} µSv/h (deviation: {deviation:+.2f}%)")

## Spectrum plots for each method

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for idx, (name, result) in enumerate(results.items()):
    ax = axes[idx]
    ax.plot(det.E_MeV, result['spectrum'], 'b-', label=f'{name}', linewidth=2)
    ax.plot(reference_spectrum['E_MeV'], reference_spectrum['ISO_ref_Cf252'], 'r--', label='Reference', linewidth=2)
    ax.set_xlabel('Energy (MeV)')
    ax.set_ylabel('Fluence')
    ax.set_title(f'{name}')
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xscale('log')
    ax.set_yscale('log')

plt.tight_layout()
plt.show()

## Metrics histogram comparison

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

x = np.arange(len(metrics_df.index))
width = 0.25

ax.bar(x - width, metrics_df['NRMSE (%)'], width, label='NRMSE (%)')
ax.bar(x, metrics_df['MAPE (%)'], width, label='MAPE (%)')
ax.bar(x + width, np.abs(metrics_df['Integral Ratio'] - 1) * 100, width, label='Integral Deviation (%)')

ax.set_xlabel('Method')
ax.set_ylabel('Error (%)')
ax.set_title('Tomography Methods - Comparison Metrics Histogram')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df.index)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Convergence behavior (if available)

In [ ]:
# Check if any methods provide convergence information
fig, ax = plt.subplots(figsize=(10, 6))

has_convergence = False
for name, result in results.items():
    if hasattr(result, 'get') and 'iterations' in result or (isinstance(result, dict) and 'history' in result):
        has_convergence = True
        # Plot convergence history if available
        pass

if not has_convergence:
    ax.text(0.5, 0.5, 'Convergence information not available\nfor these methods in current implementation',
            ha='center', va='center', transform=ax.transAxes)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')

ax.set_title('Convergence Behavior')
plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated four tomography-based reconstruction methods adapted for neutron spectrum unfolding:

1. **BSREM**: Block-Sequential Regularized EM - provides regularization during iterative updates
2. **MAP-EM**: Maximum A Posteriori EM - incorporates prior information into the reconstruction
3. **OSEM**: Ordered Subsets EM - accelerates convergence by processing subsets of data
4. **SART**: Simultaneous Algebraic Reconstruction Technique - algebraic approach from CT imaging

These methods originate from medical imaging and tomography, adapted here for the neutron spectrum unfolding problem. They offer different trade-offs between convergence speed, noise handling, and computational requirements.